## Import Datas

In [24]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/Thèse de doctorat/Redaction - rapports et articles/Articles de conférence redigés/ICATH 2026/code/data/preprocessed options datas/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [26]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
FinanceUnderlying = [ "BAC", "C", "GS", "JPM", "WFC" ]
HealthUnderlying = [ "ABBV", "JNJ", "MRK", "PFE", "UNH" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + FinanceUnderlying + HealthUnderlying + TechnologyUnderlying

In [27]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "BAC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "C" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GS" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},#
    "JPM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "WFC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "ABBV" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JNJ" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MRK" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "PFE" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "UNH" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [28]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [6]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.0 MB/s eta 0:00:00


In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

In [8]:
import optuna
import warnings

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

In [14]:
def build_ffnn(best_params, n_features):
    activations = {
        "relu": nn.ReLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
        "elu": nn.ELU,
        "selu": nn.SELU
    }

    activation = activations[best_params["activation"]]
    n_layers = best_params["n_layers"]

    layers = []
    in_dim = n_features

    for i in range(n_layers):
        out_dim = best_params[f"n_units_{i}"]
        layers.append(nn.Linear(in_dim, out_dim))
        layers.append(activation())


        in_dim = out_dim

    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers)

def train_ffnn_with_optuna(X_train, y_train, X_test, y_test, best_params, epochs=50):

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).view(-1, 1)

    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model = build_ffnn(best_params, X_train.shape[1])

    optimizer = getattr(optim, best_params["optimizer"])(
        model.parameters(),
        lr=best_params["lr"]
    )

    loss_fn = nn.MSELoss()

    # --------- Training ----------
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = loss_fn(pred, batch_y)
            loss.backward()
            optimizer.step()

    # --------- Prediction ----------
    with torch.no_grad():
        y_pred = model(X_test_t).numpy()

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    nrmse = rmse / np.mean(y_test)

    return mae, rmse, nrmse


def create_model_for_ffnn(trial, n_features):

    n_layers = trial.suggest_int("n_layers", 2, 10)# nombre de hidden layers

    activation_name = trial.suggest_categorical(
        "activation", ["relu", "tanh", "sigmoid", "elu", "selu"] # choix de la fonction d’activation
    )

    activations = {
        "relu": nn.ReLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
        "elu": nn.ELU,
        "selu": nn.SELU
    }
    activation = activations[activation_name]

    layers = []
    in_dim = n_features

    for i in range(n_layers):
        out_dim = trial.suggest_int(f"n_units_{i}", 2, 128)

        layers.append(nn.Linear(in_dim, out_dim))
        layers.append(activation())

        in_dim = out_dim

    # output layer
    layers.append(nn.Linear(in_dim, 1))

    return nn.Sequential(*layers)


# ---- objective ----
def objective_for_ffnn(trial, datasetForOptimization, X_train, y_train):

    model = create_model_for_ffnn(trial, X_train.shape[1])

    optimizer_name = trial.suggest_categorical(
        "optimizer", ["Adam", "AdamW", "SGD"] #optimizer
    )
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True) #learning_rate

    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128]) #batch_size

    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    loader = DataLoader(datasetForOptimization, batch_size=batch_size, shuffle=False)

    loss_fn = nn.MSELoss()

    # ---- training ----
    for epoch in range(40):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = loss_fn(pred, batch_y)
            loss.backward()
            optimizer.step()

    return loss.item()

def predict_current_price_using_ffnn(option_type, ticher):
  for proxy in feature_combinations:
    #Prepare training dataset
    train_dataset = dataset[ticker]["train"]
    train_dataset = train_dataset[train_dataset["optionType"] == option_type]
    X_train = train_dataset[list_histos_datas_inputs + proxy].values
    y_train = train_dataset[["lastPrice"]].values

    #Prepare test dataset
    test_dataset = dataset[ticker]["test"]
    test_dataset = test_dataset[test_dataset["optionType"] == option_type]
    X_test = test_dataset[list_histos_datas_inputs+ proxy].values
    y_test = test_dataset[["lastPrice"]].values

    #Normaliser les dataset
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.fit_transform(X_test)

    #Determine best architecture and hyper-parameters
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    datasetForOptimization = TensorDataset(X_tensor, y_tensor)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective_for_ffnn(trial, datasetForOptimization, X_train, y_train), n_trials=10)
    best_params = study.best_params
    best_params = study.best_params
    #print(best_params)

    mae, rmse, nrmse = train_ffnn_with_optuna(
        X_train, y_train,
        X_test, y_test,
        best_params
    )
    print(f"{proxy} for {ticker} => (MAE={mae:.3f}; RMSE={rmse:.3f}; ; NRMSE={nrmse:.3f})")


## Price options using xgboost

In [36]:
ticker = "GM"

In [37]:
test_size=0.2
random_state=42
target = 'lastPrice'
list_histos_datas_inputs = ["strike", "impliedVolatility", "timeToMaturity",
                            "riskFreeRate", "underlyingLastPrice"]

feature_combinations = [
    [],

    ["GS_" + ticker], ["SC_" + ticker], ["VIX"], ["PCR"],

    ["GS_" + ticker , "SC_" + ticker], ["GS_" + ticker , "VIX"], ["GS_" + ticker , "PCR"], ["SC_" + ticker , "VIX"], ["SC_" + ticker , "PCR"], ["VIX" , "PCR"],

    ["GS_" + ticker , "SC_" + ticker , "VIX"], ["GS_" + ticker , "SC_" + ticker , "PCR"], ["SC_" + ticker , "VIX" , "PCR"], ["SC_" + ticker , "VIX" , "PCR"],

	["GS_" + ticker , "SC_" + ticker , "VIX" , "PCR"]
]

In [39]:
predict_current_price_using_ffnn("put", ticker)

[] for GM => (MAE=0.330; RMSE=0.790; ; NRMSE=0.351)
['GS_GM'] for GM => (MAE=0.348; RMSE=0.601; ; NRMSE=0.267)
['SC_GM'] for GM => (MAE=0.379; RMSE=0.754; ; NRMSE=0.335)
['VIX'] for GM => (MAE=0.354; RMSE=0.776; ; NRMSE=0.345)
['PCR'] for GM => (MAE=0.397; RMSE=0.807; ; NRMSE=0.359)
['GS_GM', 'SC_GM'] for GM => (MAE=0.382; RMSE=0.652; ; NRMSE=0.290)
['GS_GM', 'VIX'] for GM => (MAE=0.351; RMSE=0.565; ; NRMSE=0.251)
['GS_GM', 'PCR'] for GM => (MAE=0.349; RMSE=0.557; ; NRMSE=0.248)
['SC_GM', 'VIX'] for GM => (MAE=0.365; RMSE=0.784; ; NRMSE=0.349)
['SC_GM', 'PCR'] for GM => (MAE=0.341; RMSE=0.682; ; NRMSE=0.303)
['VIX', 'PCR'] for GM => (MAE=0.383; RMSE=0.787; ; NRMSE=0.350)
['GS_GM', 'SC_GM', 'VIX'] for GM => (MAE=0.390; RMSE=0.613; ; NRMSE=0.272)
['GS_GM', 'SC_GM', 'PCR'] for GM => (MAE=0.388; RMSE=0.621; ; NRMSE=0.276)
['SC_GM', 'VIX', 'PCR'] for GM => (MAE=0.381; RMSE=0.749; ; NRMSE=0.333)
['SC_GM', 'VIX', 'PCR'] for GM => (MAE=0.447; RMSE=0.798; ; NRMSE=0.355)
['GS_GM', 'SC_GM', 'VIX'

In [ ]:
# [] for F => (MAE=0.220; RMSE=0.735; ; NRMSE=0.863)  ['GS_F', 'VIX'] for F => (MAE=0.210; RMSE=0.323; ; NRMSE=0.379)   ['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.244; RMSE=0.762; ; NRMSE=0.896)
# [] for GM => (MAE=0.330; RMSE=0.790; ; NRMSE=0.351) ['GS_GM', 'PCR'] for GM => (MAE=0.349; RMSE=0.557; ; NRMSE=0.248) ['GS_GM', 'SC_GM', 'VIX', 'PCR'] for GM => (MAE=0.379; RMSE=0.629; ; NRMSE=0.280)

In [ ]:
predict_current_price_using_ffnn("call", ticker)

[] for GM => (MAE=0.504; RMSE=1.208; ; NRMSE=0.166)
['GS_GM'] for GM => (MAE=0.539; RMSE=0.982; ; NRMSE=0.135)
['SC_GM'] for GM => (MAE=0.503; RMSE=1.141; ; NRMSE=0.157)
['VIX'] for GM => (MAE=0.485; RMSE=1.188; ; NRMSE=0.163)
['PCR'] for GM => (MAE=0.535; RMSE=1.227; ; NRMSE=0.169)
['GS_GM', 'SC_GM'] for GM => (MAE=0.513; RMSE=0.924; ; NRMSE=0.127)
['GS_GM', 'VIX'] for GM => (MAE=0.508; RMSE=0.891; ; NRMSE=0.123)


In [ ]:
# [] for F => (MAE=0.222; RMSE=0.739; ; NRMSE=0.868)  ['GS_F', 'VIX'] for F => (MAE=0.148; RMSE=0.211; ; NRMSE=0.248) ['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.142; RMSE=0.209; ; NRMSE=0.246)